In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("/home/jg2447/slayman/motif_inference/MOBI/scripts/")
from src import stats
from src import plt_func
from src import mobi

### data

In [ ]:
def get_inference_data_p(tomtom_summary_dir, nbp_list, rank_list, tool, top):

    # the standard to compare for significant test
    standard_spp = pd.read_csv("%s/%s_RankSPP_100_%s.txt" % (tomtom_summary_dir, tool, top), sep="\t")["accuracy"].values

    nn_data = []
    for nn in nbp_list:
        rank_data = []
        for rank in rank_list:
            data = pd.read_csv("%s/%s_%s_%d_%s.txt" % (tomtom_summary_dir, tool, rank, nn, top), sep="\t")["accuracy"].values
            p = stats.sig_test_wilcoxon(data, standard_spp)
            rank_data.append(p)
        nn_data.append(rank_data)
    
    df_p_spp = pd.DataFrame(nn_data)
    df_p_spp.columns = rank_list
    df_p_spp.index = nbp_list

    # get sig star
    df_star = df_p_spp.copy().astype(str)
    df_star[df_p_spp < 0.05] = "*"
    df_star[df_p_spp >= 0.05] = ""
    
    return(df_star)

In [ ]:
alpha_list_para = np.round(np.append(np.arange(0.1,1.0,0.1), np.arange(1.0,11.0,1.0)), decimals=2)
rank_list_para = ["RankLinear_%.1f" % i for i in alpha_list_para]
rank_list = ["RankSPP"]
rank_list.extend(rank_list_para)
rank_list.extend(["RankCrowdness"])

nbp_list = [10, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200]

tool = "DREME"
top = "top5"

df1 = get_inference_data_p("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/fly/tomtom_summary/", nbp_list, rank_list, tool, top)
df2 = get_inference_data_p("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/worm/tomtom_summary/", nbp_list, rank_list, tool, top)
df3 = get_inference_data_p("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanGM12878/tomtom_summary/", nbp_list, rank_list, tool, top)
df4 = get_inference_data_p("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanK562/tomtom_summary/", nbp_list, rank_list, tool, top)
df1.index = nbp_list
df2.index = nbp_list
df3.index = nbp_list
df4.index = nbp_list
df1.to_csv("./data_fly_p.txt", sep="\t")
df2.to_csv("./data_worm_p.txt", sep="\t")
df3.to_csv("./data_GM12878_p.txt", sep="\t")
df4.to_csv("./data_K562_p.txt", sep="\t")

df1 = mobi.get_tomtom_summary_data("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/fly/tomtom_summary/", nbp_list, rank_list, tool, top, improve=True)
df2 = mobi.get_tomtom_summary_data("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/worm/tomtom_summary/", nbp_list, rank_list, tool, top, improve=True)
df3 = mobi.get_tomtom_summary_data("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanGM12878/tomtom_summary/", nbp_list, rank_list, tool, top, improve=True)
df4 = mobi.get_tomtom_summary_data("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanK562/tomtom_summary/", nbp_list, rank_list, tool, top, improve=True)
df1.index = nbp_list
df2.index = nbp_list
df3.index = nbp_list
df4.index = nbp_list
df1.to_csv("./data_fly.txt", sep="\t")
df2.to_csv("./data_worm.txt", sep="\t")
df3.to_csv("./data_GM12878.txt", sep="\t")
df4.to_csv("./data_K562.txt", sep="\t")

### plot

In [ ]:
def plt_heatmap(data, data_p, ax, ax_cbar, vmax):

    ax = sns.heatmap(
        data, 
        ax=ax, 
        cbar_ax=ax_cbar,
        annot=data_p,
        vmin=-vmax, vmax=vmax, center=0, cmap=cmap, annot_kws={"fontsize":5}, fmt="s", linewidths=.05)

    ax.get_xaxis().set_ticks(np.arange(0.5,11,1))
    ax.get_xaxis().set_ticklabels(
        np.array(data.columns)*2,
        rotation=90, va='center', ha='right', rotation_mode="anchor", fontsize=x_tick_label_fs)
    ax.get_xaxis().set_tick_params(size=2, width=0.5, pad=1)
    ax.set_xlabel("Binding site length", fontsize=x_label_fs)

    plt_func.rm_axis_label(ax, "y")
    return(ax)

In [ ]:
figsize = (6.2, 2.7)
panel_number_fs = 8
x_tick_label_fs = 6
y_tick_label_fs = 6
x_label_fs = 7
y_label_fs = 7
title_fs = 7
legend_fs = 6

cmap = "RdBu"

# read data
df_fly = pd.read_csv("./data_fly.txt", sep="\t", index_col=0).T
df_fly_p = pd.read_table("./data_fly_p.txt", sep="\t", index_col=0, na_filter=False).T
df_worm = pd.read_csv("./data_worm.txt", sep="\t", index_col=0).T
df_worm_p = pd.read_table("./data_worm_p.txt", sep="\t", index_col=0, na_filter=False).T
df_GM12878 = pd.read_csv("./data_GM12878.txt", sep="\t", index_col=0).T
df_GM12878_p = pd.read_table("./data_GM12878_p.txt", sep="\t", index_col=0, na_filter=False).T
df_K562 = pd.read_csv("./data_K562.txt", sep="\t", index_col=0).T
df_K562_p = pd.read_table("./data_K562_p.txt", sep="\t", index_col=0, na_filter=False).T

vmax = pd.concat([df_fly, df_worm, df_GM12878, df_K562]).abs().max().max()

sns.set_context("paper")

fig = plt.figure(figsize=figsize, dpi=300)
gs = mpl.gridspec.GridSpec(3,9, width_ratios=[3, 11, 0.15, 11, 0.15, 11, 0.15, 11, 0.5], wspace=0.15) 

ax_cbar = fig.add_subplot(gs[1, 8])

#### A
ax = fig.add_subplot(gs[:, 1])

plt_heatmap(df_fly, df_fly_p, ax, ax_cbar, vmax)
ax.set_title("Fly", fontsize=title_fs)
ax.annotate("★", (5.05, 0.8), fontsize=6, color='red')
ax.text(-0.2, 1.05, "A", transform=ax.transAxes, size=panel_number_fs, weight='bold')

ax.get_yaxis().set_ticks(np.arange(0.5,21,1))
ax.get_yaxis().set_ticklabels(["Signal","0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1.0", "2.0", "3.0", "4.0", "5.0", "6.0", "7.0", "8.0", "9.0", "10.0","C-score"],  
                              fontsize=y_tick_label_fs)
ax.get_yaxis().set_tick_params(size=2, width=0.5, pad=1)
ax.set_ylabel("Binding site ranking method", fontsize=y_label_fs, labelpad=-0.2)

##### BC-score label
ax_annot = fig.add_subplot(gs[:, 0])
ax_annot.plot([1,1], [0,21], ",", ms=0)
ax_annot.set_ylim([0,21])
ax_annot.set_xlim([0,2])    
ax_annot.annotate("", 
                  (0.4, 1.5),
                  (0.4, 4),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (0.4, 17),
                  (0.4, 19.5),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (0, 1.5),
                  (0.8, 1.5),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (0, 19.5),
                  (0.8, 19.5),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("BC-score with different weight", 
                  (0.2, 4.1),
                  (0.2, 4.1),
                  rotation=90,
                  size=6)
ax_annot.axis('off')


#### B
ax = fig.add_subplot(gs[:, 3])
plt_heatmap(df_worm, df_worm_p, ax, ax_cbar, vmax)
ax.set_title("Worm", fontsize=title_fs)
ax.annotate("★", (5.05, 0.8), fontsize=6, color='red')
ax.text(-0.1, 1.05, "B", transform=ax.transAxes, size=panel_number_fs, weight='bold')

#### C
ax = fig.add_subplot(gs[:, 5])
plt_heatmap(df_GM12878, df_GM12878_p, ax, ax_cbar, vmax)
ax.set_title("GM12878", fontsize=title_fs)
ax.annotate("★", (5.05, 0.8), fontsize=6, color='red')
ax.text(-0.1, 1.05, "C", transform=ax.transAxes, size=panel_number_fs, weight='bold')

#### D
ax = fig.add_subplot(gs[:, 7])
plt_heatmap(df_K562, df_K562_p, ax, ax_cbar, vmax)
ax.set_title("K562", fontsize=title_fs)
ax.annotate("★", (5.05, 0.8), fontsize=6, color='red')
ax.text(-0.1, 1.05, "D", transform=ax.transAxes, size=panel_number_fs, weight='bold')

####
# color bar ticks
ax_cbar.yaxis.set_ticks([-0.8, -0.4, 0, 0.4, 0.8])
ax_cbar.yaxis.set_tick_params(size=2, width=0.5)
ax_cbar.tick_params(axis='y', which='major', pad=2, labelsize=legend_fs)

####
plt.savefig("./fig3.pdf", dpi="figure", bbox_inches="tight")